<a href="https://colab.research.google.com/github/seleneyong/ISYS2001-MyWork/blob/main/Copy_of_1_financial_chatbot_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevin-blasiak-curtin/ISYS2001-Archive/blob/main/Module%2008%20-%20API/1_financial_chatbot_gemini.ipynb)

# Building a Financial Advisor Chatbot

**Talking to an AI model through the Gemini API**

## What you'll build

A small chatbot we'll call the Financial Sage. It answers everyday money questions in plain language, and it can read a file of transactions and talk about your spending.

By the end you will have sent your first request to a real AI service and got a reply back in your own code.

## The one idea behind this week

An API is a request-and-response contract. You send a request, and something on the other side sends back a structured response you can use.

You will meet two APIs over the next two tasks:

- **Gemini** (this worksheet): you send it text, it sends text back.
- **A stock service** (the lab ticket): you send it a company code, it sends back numbers you can chart.

Same shape, different contents. Keep that picture in your head and the rest is detail.

## Setup

In [1]:
# The Google Gen AI library is how our Python code talks to Gemini.
!pip install google-genai

### Get your own API key

An API key is a private password that proves you are allowed to use the service. You get your own free one from Google AI Studio.

1. Go to https://aistudio.google.com/api-keys and sign in with the Google account you already use for Colab.
2. Create an API key and copy it.
3. In Colab, open the Secrets panel (the key icon in the left sidebar).
4. Add a new secret named exactly `GOOGLE_API_KEY`, paste your key into the value, and switch **Notebook access** on.

The name in the Secrets panel must match the name in the code, character for character. If it does not, you will get a "secret not found" error.

We store the key this way so it never sits inside a code cell where it could be shared or copied by accident.

In [2]:
from google.colab import userdata
from google import genai

# The client reads your key from Secrets, so the key itself never appears here.
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

# The flash model is fast and free-tier friendly. The exact name changes every
# few months, so check AI Studio for the current one and edit this single line.
MODEL = "gemini-3.5-flash"

print("Connected to the Gemini API.")

SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.

### One small helper

Every time we want the model to answer something, we send a prompt and read the reply. Rather than write that out each time, we wrap it in a function once.

In [ ]:
def ask_gemini(prompt):
    """Send a prompt to Gemini and return the text of its reply."""
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text
    except Exception as error:
        return f"Something went wrong talking to the API: {error}"


# A quick test. If your key is set up, you should get a one-line answer back.
print(ask_gemini("In one sentence, what is compound interest?"))

## Step 1: Think before you build

Before writing the chatbot, it helps to plan it. This is a good moment to use AI as a thinking partner. Try a prompt like this in AI Studio or Colab's assistant:

```
Help me plan a simple financial advisor chatbot that:
1. answers everyday money questions with general, educational information
2. can read transaction data from a CSV file
3. speaks in a friendly, plain tone
4. always reminds the user this is not professional advice

What functions would I need, and what should each one do?
```

Notice we are asking it to help us plan, not to write the whole thing. The plan is ours.

## Step 2: The Financial Sage

In [ ]:
def financial_sage(question):
    """Answer a money question in the voice of a friendly, cautious guide."""
    personality = (
        "You are a friendly financial guide for university students. "
        "You give general, educational information about budgeting, saving "
        "and everyday money decisions. You keep answers short and plain. "
        "You always remind the reader that this is general information, "
        "not professional financial advice."
    )

    prompt = personality + "\n\nQuestion: " + question
    return ask_gemini(prompt)


# Try it
print(financial_sage("Should I keep my savings in a bank account or invest it?"))

## Step 3: Some data to talk about

Let's make a small transactions file. Income is positive, spending is negative. We save it once so the rest of the notebook can read it.

In [ ]:
import pandas as pd

transactions = {
    "Date": ["2024-01-01", "2024-01-05", "2024-01-06", "2024-01-07",
             "2024-01-10", "2024-01-12", "2024-01-15", "2024-01-18",
             "2024-01-20", "2024-01-22", "2024-01-25"],
    "Description": ["Salary", "Groceries Coles", "Electricity Bill", "Coffee Shop",
                    "Gym Membership", "Restaurant Dinner", "Rent Payment", "Movie Tickets",
                    "Groceries Coles", "Petrol", "Streaming Service"],
    "Category": ["Income", "Food", "Utilities", "Food",
                 "Health", "Dining", "Housing", "Entertainment",
                 "Food", "Transport", "Entertainment"],
    "Amount": [3000.00, -45.50, -120.00, -5.50,
               -50.00, -65.00, -1200.00, -30.00,
               -52.30, -60.00, -15.99],
}

df = pd.DataFrame(transactions)
df.to_csv("transactions.csv", index=False)
df

## Step 4: Turn the data into a summary

Before the model can say anything useful about your spending, we need to hand it a tidy summary rather than the raw rows. This function does the arithmetic with pandas and returns a short block of text.

In [ ]:
def analyse_transactions(csv_file):
    """Read a transactions file and return a short written summary."""
    df = pd.read_csv(csv_file)

    total_income = df[df["Amount"] > 0]["Amount"].sum()
    total_expenses = abs(df[df["Amount"] < 0]["Amount"].sum())
    net_savings = total_income - total_expenses
    savings_rate = (net_savings / total_income * 100) if total_income > 0 else 0

    by_category = df[df["Amount"] < 0].groupby("Category")["Amount"].sum().abs()

    summary = f"""Financial summary:
- Total income: ${total_income:.2f}
- Total expenses: ${total_expenses:.2f}
- Net savings: ${net_savings:.2f}
- Savings rate: {savings_rate:.1f}%

Spending by category:
{by_category.to_string()}"""

    return summary


print(analyse_transactions("transactions.csv"))

## Step 5: Advice based on the data

Now we join the two ideas. We give the model the summary and a question at the same time, so its answer is grounded in the actual numbers.

In [ ]:
def get_advice(csv_file, question):
    """Combine the transaction summary with a question and ask the sage."""
    summary = analyse_transactions(csv_file)

    prompt = f"""Here is the person's financial data:
{summary}

Their question: {question}

Give short, general, educational guidance based on this data."""

    return financial_sage(prompt)


print(get_advice("transactions.csv",
                  "Where is most of my money going, and what could I look at first?"))

## Step 6: A simple chat loop

This ties it together into something you can talk to. Type `analyse` to see your summary, ask any money question, or type `quit` to stop.

In [ ]:
def chat_with_sage():
    """A small interactive loop over the sage."""
    print("Financial Sage is ready.")
    print("Ask a money question, type 'analyse' to look at your transactions, or 'quit' to stop.")
    print()

    while True:
        user_input = input("You: ")

        if user_input.lower() == "quit":
            print("Sage: Take care with your money. Goodbye.")
            break
        elif user_input.lower() == "analyse":
            print("Sage: Here is your summary:")
            print(analyse_transactions("transactions.csv"))
        else:
            print("Sage:", get_advice("transactions.csv", user_input))

        print()

In [ ]:
chat_with_sage()

## Step 7: Try a few questions at once

A quick way to see how the sage handles different kinds of question. Questions that mention your spending get routed through the data; the rest get general answers.

In [ ]:
questions = [
    "I have just started a part-time job. How should I think about saving?",
    "What is the difference between a debit card and a credit card?",
    "Based on my spending, is there anything I should watch?",
]

for q in questions:
    print("Question:", q)
    if "based on my spending" in q.lower():
        answer = get_advice("transactions.csv", q)
    else:
        answer = financial_sage(q)
    print("Sage:", answer[:300], "...")
    print("-" * 40)

## Your turn

Pick one small feature and build it, using AI as a thinking partner if you get stuck. You must be able to explain every line you keep.

Ideas:
- a savings goal tracker
- a "biggest three expenses" report
- a monthly budget check

Reflection prompt worth trying: paste your finished chatbot to an AI and ask, "What could go wrong with this, and what would you check before trusting its advice?"

In [ ]:
# Your feature here.
def my_feature():
    pass

## Where we got to

You have:

- sent real requests to the Gemini API and read the replies
- kept your key out of your code using Colab Secrets
- combined pandas data work with an AI model to give grounded answers

The stock lab ticket uses the same request-and-response idea, only the response is numbers instead of text.

A reminder: this is educational content, not professional financial advice.